# Notebook 5: 평가 재설계 — 합성 구조 이상 벤치마크

## 왜 이 노트북이 필요한가?

NB04의 평가는 "구조 이상 탐지기"를 SlideAudit(디자인 결함 라벨)로 측정한다.
구조 이상(역할 순서 붕괴, 도입부 누락)과 디자인 결함(폰트/색상)은 개념이 다르므로
완벽한 탐지기도 AUC ≈ 0.5가 나올 수 있다.

이 노트북은 정상 덱에 통제된 변형을 가해 ground truth를 직접 생성하고,
HMM 탐지기가 실제로 구조 이상을 인식하는지 검증한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

In [ ]:
!pip install -q scipy
print('설치 완료')

## 1. 합성 이상 함수 정의

3가지 이상 유형 × mild/severe = 6가지 변형.

In [ ]:
import random
import copy
import numpy as np
import pandas as pd
import ast, json, pickle
from pathlib import Path
from sklearn.metrics import roc_auc_score

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']


def augment_shuffle_mild(sequence: list, seed: int = None) -> list:
    """중간 슬라이드(표지/마무리 제외)만 순서를 섞는다."""
    seq = copy.deepcopy(sequence)
    if len(seq) <= 3:
        return seq
    rng = random.Random(seed)
    middle = seq[1:-1]
    rng.shuffle(middle)
    return [seq[0]] + middle + [seq[-1]]


def augment_shuffle_severe(sequence: list, seed: int = None) -> list:
    """표지/마무리 포함 전체 순서를 섞는다."""
    seq = copy.deepcopy(sequence)
    if len(seq) <= 2:
        return seq
    rng = random.Random(seed)
    rng.shuffle(seq)
    return seq


def augment_remove_boundary(sequence: list, side: str = 'both') -> list:
    """표지(start), 마무리(end), 또는 둘 다 제거한다."""
    seq = copy.deepcopy(sequence)
    if side in ('start', 'both') and len(seq) > 2:
        seq = seq[1:]
    if side in ('end', 'both') and len(seq) > 2:
        seq = seq[:-1]
    return seq


def augment_no_cover(sequence: list) -> list:
    return augment_remove_boundary(sequence, side='start')


def augment_no_closing(sequence: list) -> list:
    return augment_remove_boundary(sequence, side='end')


def augment_duplicate_section(sequence: list, dup_ratio: float = 0.3, seed: int = None) -> list:
    """중간 슬라이드 일부를 표지 직후에 삽입 (섹션 반복 이상)."""
    seq = copy.deepcopy(sequence)
    if len(seq) <= 4:
        return seq
    rng = random.Random(seed)
    middle = seq[1:-1]
    n_dup = max(1, int(len(middle) * dup_ratio))
    dup = rng.sample(middle, n_dup)
    return [seq[0]] + dup + seq[1:]


print('이상 증강 함수 정의 완료')

## 2. 합성 벤치마크 생성

정상 덱에서 1:1 비율로 이상 샘플 생성. 6가지 이상 유형별로 균등 분배.

In [ ]:
seq_df = pd.read_csv(f'{LABELS_DIR}/sequences_cnn.csv')
seq_df['sequence'] = seq_df['sequence'].apply(ast.literal_eval)

# 최소 길이 4 이상인 덱만 사용 (너무 짧은 덱은 의미 있는 이상 생성 불가)
seq_df = seq_df[seq_df['length'] >= 4].reset_index(drop=True)
print(f'사용 가능한 덱: {len(seq_df)}개')

AUGMENT_TYPES = [
    ('shuffle_mild',   lambda seq, i: augment_shuffle_mild(seq, seed=i)),
    ('shuffle_severe', lambda seq, i: augment_shuffle_severe(seq, seed=i)),
    ('no_cover',       lambda seq, i: augment_no_cover(seq)),
    ('no_closing',     lambda seq, i: augment_no_closing(seq)),
    ('no_boundary',    lambda seq, i: augment_remove_boundary(seq, 'both')),
    ('dup_section',    lambda seq, i: augment_duplicate_section(seq, seed=i)),
]

# 이상 유형별로 균등하게 샘플 수를 배분
n_per_type = len(seq_df) // len(AUGMENT_TYPES)
records = []

# 정상 샘플 전부 추가
for _, row in seq_df.iterrows():
    records.append({
        'deck_id':          row['deck_id'] + '_normal',
        'original_deck_id': row['deck_id'],
        'sequence':         row['sequence'],
        'anomaly_type':     'normal',
        'is_anomaly':       0,
    })

# 이상 샘플 추가
for aug_type, aug_fn in AUGMENT_TYPES:
    subset = seq_df.sample(n=n_per_type, random_state=42).reset_index(drop=True)
    for i, row in subset.iterrows():
        aug_seq = aug_fn(row['sequence'], i)
        if aug_seq == row['sequence']:
            continue  # 변형 없으면 skip
        records.append({
            'deck_id':          f'{row["deck_id"]}_anomaly_{aug_type}',
            'original_deck_id': row['deck_id'],
            'sequence':         aug_seq,
            'anomaly_type':     aug_type,
            'is_anomaly':       1,
        })

benchmark_df = pd.DataFrame(records)
benchmark_df.to_csv(f'{LABELS_DIR}/synthetic_anomaly_benchmark.csv', index=False)

print(f'벤치마크 생성 완료: {len(benchmark_df)}개')
print(f'  정상: {(benchmark_df["is_anomaly"]==0).sum()}개')
print(f'  이상: {(benchmark_df["is_anomaly"]==1).sum()}개')
print(f'\n이상 유형별 분포:')
print(benchmark_df[benchmark_df['is_anomaly']==1]['anomaly_type'].value_counts())

## 3. HMM 탐지기로 합성 벤치마크 평가

In [ ]:
with open(f'{MODELS_DIR}/hmm_model.pkl', 'rb') as f:
    hmm_model = pickle.load(f)
with open(f'{MODELS_DIR}/hmm_thresholds.json') as f:
    thresholds = json.load(f)

scores = []
for _, row in benchmark_df.iterrows():
    seq = np.array(row['sequence']).reshape(-1, 1)
    if len(seq) < 5:
        scores.append(0.5)
        continue
    ll = hmm_model.score(seq) / len(seq)
    z = (thresholds['mean'] - ll) / (thresholds['std'] + 1e-8)
    scores.append(float(np.clip(z / 3.0, 0, 1)))

benchmark_df['hmm_score'] = scores
labels = benchmark_df['is_anomaly'].values
auc_synthetic = roc_auc_score(labels, scores)

print(f'합성 벤치마크 AUC (HMM): {auc_synthetic:.4f}')
print()

# 이상 유형별 AUC
for aug_type, _ in AUGMENT_TYPES:
    subset = benchmark_df[benchmark_df['anomaly_type'].isin(['normal', aug_type])]
    if subset['is_anomaly'].sum() == 0:
        continue
    type_auc = roc_auc_score(subset['is_anomaly'], subset['hmm_score'])
    print(f'  {aug_type}: AUC={type_auc:.4f}')

## 4. SlideAudit AUC vs 합성 벤치마크 AUC 비교

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

# SlideAudit AUC는 NB04에서 가져온다 (파일 없으면 None 처리)
final_results_path = Path(f'{MODELS_DIR}/final_results.json')
if final_results_path.exists():
    with open(final_results_path) as f:
        nb04_results = json.load(f)
    slideaudit_auc = nb04_results.get('new_pipeline_auc', None)
else:
    slideaudit_auc = None
    print('⚠ final_results.json 없음 — NB04를 먼저 실행하라')

comparison = {
    'slideaudit_auc':      slideaudit_auc,
    'synthetic_auc_hmm':   float(auc_synthetic),
    'interpretation': (
        'synthetic_auc >> slideaudit_auc: 모델은 구조 이상을 탐지하지만 SlideAudit 평가가 부적절'
        if (slideaudit_auc and auc_synthetic - slideaudit_auc > 0.10)
        else 'synthetic_auc ≈ slideaudit_auc: weak label 문제로 HMM 자체가 구조 인식 못함'
        if auc_synthetic < 0.60
        else '참고용 — 추가 분석 필요'
    ),
}
print(json.dumps(comparison, indent=2, ensure_ascii=False))

with open(f'{MODELS_DIR}/eval_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)

## 5. 이상 유형별 ROC 커브 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 좌: 유형별 AUC 바 차트
type_aucs = {}
for aug_type, _ in AUGMENT_TYPES:
    subset = benchmark_df[benchmark_df['anomaly_type'].isin(['normal', aug_type])]
    if subset['is_anomaly'].sum() == 0:
        continue
    type_aucs[aug_type] = roc_auc_score(subset['is_anomaly'], subset['hmm_score'])

colors = ['steelblue' if v > 0.6 else 'tomato' for v in type_aucs.values()]
axes[0].barh(list(type_aucs.keys()), list(type_aucs.values()), color=colors)
axes[0].axvline(0.5, color='red', linestyle='--', label='Random')
axes[0].set_xlabel('AUC')
axes[0].set_title('이상 유형별 HMM AUC')
axes[0].set_xlim(0, 1)
axes[0].legend()

# 우: 전체 ROC 커브
fpr, tpr, _ = roc_curve(labels, scores)
axes[1].plot(fpr, tpr, label=f'HMM (AUC={auc_synthetic:.3f})', color='steelblue')
axes[1].plot([0,1],[0,1],'k:', label='Random')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('합성 벤치마크 전체 ROC')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/synthetic_benchmark_roc.png', dpi=120)
plt.show()
print('저장 완료: synthetic_benchmark_roc.png')

In [ ]:
print('=== Notebook 5 완료 ===')
print(f'합성 벤치마크: {LABELS_DIR}/synthetic_anomaly_benchmark.csv')
print(f'평가 비교: {MODELS_DIR}/eval_comparison.json')
print(f'HMM 합성 벤치마크 AUC: {auc_synthetic:.4f}')
if auc_synthetic < 0.6:
    print('\n⚠ AUC < 0.6: HMM이 구조를 인식하지 못함 → NB06 CLIP weak label로 개선 필요')
elif auc_synthetic > 0.7:
    print('\n✓ AUC > 0.7: HMM이 구조 이상을 탐지함 → SlideAudit 평가가 부적절했음을 의미')
print('\nNotebook 6 (CLIP Weak Label)으로 이동하세요.')